# 07_eval / newsclippings_eval.ipynb

**Purpose:** Formal in-distribution evaluation of the GSAF fusion model on **NewsCLIPpings test** (the primary benchmark our model was trained for).

**Inputs:**
- `cache/fusion/fusion_predictions.csv` — per-row predictions written by `05_fusion/fusion_training.ipynb`
- `cache/fusion/fusion_thresholds.json` — per-dataset tuned thresholds
- `data/processed/ooc_pairs/manifests/all.csv` — for source dataset/split labels
- `cache/stream_a/results/stream_a_predictions.csv`, `cache/stream_b/final/stream_b_predictions.csv`, `cache/stream_c/results/stream_c_predictions.csv` — for per-stream comparison

**Outputs:**
- `paper/tables/newsclippings_eval.csv` — full metric table (fusion + 3 streams + 4 ablations) on NewsCLIPpings test
- `paper/tables/newsclippings_eval_per_class.csv` — per-class precision/recall/F1
- `paper/figures/newsclippings_eval_metric_bars.png` — grouped bar chart of AUROC / F1 / Acc / AUPRC

**Notes:**
- This notebook **does not retrain anything** — it only loads pre-computed predictions.
- Threshold for binary metrics comes from `fusion_thresholds.json` (tuned on val) — never on test.
- If a column is missing, the cell will print a warning and skip (no crash).


In [ ]:
# Imports & path setup
from __future__ import annotations
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score, classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Add repo root to sys.path so we can import src.utils.paths
NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, STREAM_C_CACHE,
    PAPER, FIGURES, TABLES,
)

print("Repo root:", ROOT)
print("Fusion preds:", FUSION_CACHE / "fusion_predictions.csv")
print("Tables out :", TABLES)
print("Figures out:", FIGURES)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)


## 1. Load fusion predictions and thresholds

In [ ]:
fusion_path = FUSION_CACHE / "fusion_predictions.csv"
thresh_path = FUSION_CACHE / "fusion_thresholds.json"
assert fusion_path.exists(), f"Missing {fusion_path}. Run 05_fusion/fusion_training.ipynb first."

fusion_df = pd.read_csv(fusion_path)
print("fusion_predictions shape:", fusion_df.shape)
print("Columns:", list(fusion_df.columns))
fusion_df.head()


In [ ]:
# Thresholds
if thresh_path.exists():
    with open(thresh_path) as f:
        thresholds = json.load(f)
    print("Loaded thresholds:", json.dumps(thresholds, indent=2))
else:
    print("No threshold file found; falling back to 0.5 everywhere.")
    thresholds = {}

DATASET_KEY_NC = "newsclippings_test"
THRESH_NC = float(thresholds.get(DATASET_KEY_NC, thresholds.get("default", 0.5)))
print(f"NewsCLIPpings test threshold = {THRESH_NC:.4f}")


## 2. Filter to NewsCLIPpings test split

In [ ]:
# We need to identify rows belonging to NewsCLIPpings test.
# fusion_predictions.csv should have a 'dataset' (or 'source_dataset') and 'split' column,
# but if not we join against ALL_MANIFEST on pair_id.

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

ds_col    = find_col(fusion_df, ["dataset", "source_dataset", "src_dataset"])
split_col = find_col(fusion_df, ["split", "data_split"])
pid_col   = find_col(fusion_df, ["pair_id", "id"])
y_col     = find_col(fusion_df, ["y_true", "label", "target"])
p_col     = find_col(fusion_df, ["fusion_prob", "y_prob_fusion", "prob", "y_prob"])

print(f"dataset col: {ds_col}, split col: {split_col}, pair_id col: {pid_col}")
print(f"y_true col : {y_col}, fusion prob col: {p_col}")

if ds_col is None or split_col is None:
    # Join with manifest
    print("\nJoining with all.csv to recover dataset/split…")
    manifest = pd.read_csv(ALL_MANIFEST, usecols=lambda c: c in {"pair_id", "dataset", "split"})
    fusion_df = fusion_df.merge(manifest, on="pair_id", how="left", suffixes=("", "_manifest"))
    ds_col = "dataset"
    split_col = "split"

# Filter
nc_test = fusion_df[(fusion_df[ds_col] == "newsclippings") & (fusion_df[split_col] == "test")].copy()
print(f"\nNewsCLIPpings test rows: {len(nc_test):,}")
nc_test.head()


## 3. Metric helper

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, name="model"):
    """Return dict of binary classification metrics."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    mask = ~np.isnan(y_prob) & ~np.isnan(y_true)
    y_true, y_prob = y_true[mask], y_prob[mask]
    if len(np.unique(y_true)) < 2:
        return {"model": name, "n": int(mask.sum()), "auroc": np.nan, "auprc": np.nan,
                "accuracy": np.nan, "f1": np.nan, "precision": np.nan, "recall": np.nan,
                "threshold": threshold}
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "model":     name,
        "n":         int(mask.sum()),
        "auroc":     roc_auc_score(y_true, y_prob),
        "auprc":     average_precision_score(y_true, y_prob),
        "accuracy":  accuracy_score(y_true, y_pred),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
    }


## 4. Fusion model and ablations on NewsCLIPpings test

In [ ]:
# Try common column names for fusion + ablation probability outputs
candidate_cols = {
    "GSAF (full)":        ["fusion_prob", "prob_fusion", "y_prob_fusion", "y_prob"],
    "Ablation: no Stream A":   ["prob_no_a", "ab_no_a_prob", "y_prob_no_a"],
    "Ablation: no Stream C":   ["prob_no_c", "ab_no_c_prob", "y_prob_no_c"],
    "Ablation: MLP fusion":    ["prob_mlp_fusion", "ab_mlp_prob", "y_prob_mlp"],
    "Ablation: frozen CLIP":   ["prob_frozen_clip", "ab_frozen_prob", "y_prob_frozen_clip"],
    "Ablation: no MLLM":       ["prob_no_mllm", "ab_no_mllm_prob", "y_prob_no_mllm"],
}

rows = []
for label, cands in candidate_cols.items():
    col = find_col(nc_test, cands)
    if col is None:
        print(f"[skip] {label}: no matching column in fusion_predictions.csv (tried {cands})")
        continue
    m = compute_metrics(nc_test[y_col], nc_test[col], threshold=THRESH_NC, name=label)
    rows.append(m)

fusion_metrics_df = pd.DataFrame(rows)
fusion_metrics_df


## 5. Per-stream baselines on NewsCLIPpings test

In [ ]:
stream_files = {
    "Stream A (entities)":  STREAM_A_CACHE / "results" / "stream_a_predictions.csv",
    "Stream B (CLIP)":      STREAM_B_CACHE / "final"   / "stream_b_predictions.csv",
    "Stream C (NLI)":       STREAM_C_CACHE / "results" / "stream_c_predictions.csv",
}

stream_rows = []
for name, fp in stream_files.items():
    if not fp.exists():
        print(f"[skip] {name}: file not found at {fp}")
        continue
    sdf = pd.read_csv(fp)
    sds_col = find_col(sdf, ["dataset", "source_dataset"])
    ssplit_col = find_col(sdf, ["split", "data_split"])
    sy_col = find_col(sdf, ["y_true", "label", "target"])
    sp_col = find_col(sdf, ["y_prob", "prob", "score", "stream_prob"])

    if sds_col is None or ssplit_col is None:
        # Join with manifest
        manifest = pd.read_csv(ALL_MANIFEST, usecols=lambda c: c in {"pair_id", "dataset", "split"})
        sdf = sdf.merge(manifest, on="pair_id", how="left")
        sds_col, ssplit_col = "dataset", "split"

    sub = sdf[(sdf[sds_col] == "newsclippings") & (sdf[ssplit_col] == "test")].copy()
    if sub.empty or sy_col is None or sp_col is None:
        print(f"[skip] {name}: empty subset or missing prob/label cols")
        continue
    m = compute_metrics(sub[sy_col], sub[sp_col], threshold=0.5, name=name)
    stream_rows.append(m)

stream_metrics_df = pd.DataFrame(stream_rows)
stream_metrics_df


## 6. Combine, save, and display

In [ ]:
all_metrics = pd.concat([stream_metrics_df, fusion_metrics_df], ignore_index=True)
all_metrics = all_metrics[["model", "n", "auroc", "auprc", "accuracy", "f1", "precision", "recall", "threshold"]]
out_csv = TABLES / "newsclippings_eval.csv"
all_metrics.to_csv(out_csv, index=False, float_format="%.4f")
print(f"Saved: {out_csv}")
all_metrics.style.format({c: "{:.4f}" for c in ["auroc","auprc","accuracy","f1","precision","recall","threshold"]})


## 7. Per-class breakdown for the full GSAF fusion

In [ ]:
fusion_col = find_col(nc_test, candidate_cols["GSAF (full)"])
if fusion_col is not None:
    y_true = nc_test[y_col].astype(int).values
    y_pred = (nc_test[fusion_col].values >= THRESH_NC).astype(int)
    rep = classification_report(y_true, y_pred, target_names=["genuine (0)", "OOC (1)"],
                                digits=4, output_dict=True, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_df.to_csv(TABLES / "newsclippings_eval_per_class.csv", float_format="%.4f")
    print("Per-class report saved to newsclippings_eval_per_class.csv")
    print()
    print(classification_report(y_true, y_pred, target_names=["genuine (0)", "OOC (1)"], digits=4, zero_division=0))
    print("Confusion matrix [rows=true, cols=pred]:")
    print(confusion_matrix(y_true, y_pred))
else:
    print("Fusion column missing; skipping per-class report.")


## 8. Bar plot of metrics

In [ ]:
plot_df = all_metrics.dropna(subset=["auroc"]).copy()
metric_cols = ["auroc", "auprc", "accuracy", "f1"]
n_models = len(plot_df)
n_metrics = len(metric_cols)

fig, ax = plt.subplots(figsize=(max(8, 1.2 * n_models), 5))
x = np.arange(n_models)
width = 0.8 / n_metrics

for i, m in enumerate(metric_cols):
    ax.bar(x + i * width - 0.4 + width / 2, plot_df[m].values, width=width, label=m.upper())

ax.set_xticks(x)
ax.set_xticklabels(plot_df["model"].values, rotation=20, ha="right")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("NewsCLIPpings test: in-distribution metrics")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right", ncol=2, fontsize=9)
plt.tight_layout()

out_fig = FIGURES / "newsclippings_eval_metric_bars.png"
plt.savefig(out_fig, bbox_inches="tight", dpi=150)
print(f"Saved: {out_fig}")
plt.show()


---
**Done.** This notebook produced the in-distribution NewsCLIPpings test table and bar plot. The cross-dataset (OOD) evaluation is in `cross_dataset_eval.ipynb`.